In [409]:
import import_ipynb
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('punkt_tab')
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn
from nltk.translate.bleu_score import sentence_bleu
import spacy
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\krist\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [410]:
#Import functions from other notebook
from nlp_processing import text_preprocessing, nlp_processing, load_ingredients_list, tag_ingredients, identify_ingredients, perform_tfidf, perform_cosine_similarity

In [411]:
def check_valid_ingredients(ingredient, ingredients_list):
    #Check if any valid ingredients were added
    if ingredient is None:
        print("Ingredient was not recognized. Please try again")
    #Check if ingredient is already on the list
    elif ingredient in ingredients_list:
         print("You have already added:", ingredient)
    else:
        #If valid add the ingredient to the user's ingredient list
        ingredients_list.append(ingredient)
        print("Current Ingredients List:", ingredients_list)

In [412]:
#Lemmatized verbs or commands e.g.:finished->finish
def lemmatized_verbs(token):
    wn = nltk.WordNetLemmatizer()
    text_lemmatized = []
    for word,tag in pos_tag(token):
        #Check if the token/word has a POS verb tag
        if tag[0] == "V" and word != "done":
             text_lemmatized.append(wn.lemmatize(word, tag[0].lower())) #Use the "v" tag to indicate that its a verb
        else:
            text_lemmatized.append(wn.lemmatize(word))
    return text_lemmatized

In [420]:
#Check if a token is synonymous/matches a specific command such as "finished" or "done"
def check_valid_commands(tokens, status):
    # Using Word2 net to find synonyms of the word "finish"
    command = "finish"
    synonyms = []
    for synonym in wn.synsets(command):
        for lemma in synonym.lemma_names():
            synonyms.append(lemma)
    synonyms.append("done") #Also add done to the synonym list
    
    #Ensure that the commands are lemmatized
    tokens = lemmatized_verbs(tokens)
    #Check if the user's token matches that of any word in the synonym list
    for token in tokens:
        if token in synonyms:
            status = "done"
            return status

In [421]:
def convert_words2number(token):
    words_to_nums = {
        "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
        "five": "5", "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5"
    }
    return words_to_nums.get(token)

In [422]:
#using Bleu score to match the recipe name given by the user to one of the top 5 recipe list
def get_bleu_score(self, tokens):
    tokens = [tokens]
    #Get the index positions of the top 5 recipes
    index = self.top5_recipe.index
    chosen_index = 0
    #Calculate the bleu score for each recipe list
    bleu_score = 0
    for i, recipe in enumerate(self.top5_recipe):
        #Perform text cleaning on recipe name
        recipe = text_preprocessing(recipe)
        recipe = word_tokenize(recipe)
        #Calculate the bleu score
        score = sentence_bleu(tokens, recipe)
        if score > bleu_score:
            bleu_score = score
            chosen_index = index[i]
            
    #If no matches found return
    if bleu_score == 0:
        return
    #Otherwise return the recipe's index with the highest bleu score
    else:
        return chosen_index

In [423]:
#
def check_recipe_response(tokens, self):
    #First, check if user used numbers/digit to pick a recipe from the list
    for token in tokens:
        #Check for worded numbers and convert them to digits
        number_choice = convert_words2number(token)
        if number_choice is not None:
            token = number_choice
        #Check for numbers/digits and check if this number is within range of 1-5
        if token.isnumeric() and int(token) in range(1,6):
            index = self.top5_recipe.index
            token = int(token) #Convert to int
            chosen_index = index[token - 1]
            #Return the dataframe index of the user's choice
            return chosen_index
    #Second, if user entered a name instead find a matchin one based on Bleu score   
    chosen_index = get_bleu_score(self, tokens)
    return chosen_index

In [424]:
#Print Chosen Recipe

In [427]:
#Dialogue Policy
class DialoguePolicy:
    def __init__(self):
        self.ingredients_list = []
        self.top5_recipe = []
        self.current_stage = "ingredients_collection"
        #Import the food dataset
        self.df_food = pd.read_csv("./dataset/Recipe_Dataset.csv")

    ####################3 main stages: ingredients collection, recipe choosing, recipe output
    ###Step 1.) Ingredients Collection Stage
    def choose_ingredient_stage(self):
        status = "ongoing"
        while status != "done":
            #Get user response
            response = input("Choose your ingredients:")
            #Apply text cleaning to response
            response = text_preprocessing(response)
            response,tokens = nlp_processing(response)
            #Check if the user's response contains a command
            status = check_valid_commands(tokens, status)
            if status == "done":
                break
            #Detect ingredients from text
            all_ingredients = load_ingredients_list()
            matcher, nlp = tag_ingredients(all_ingredients)
            ingredient = identify_ingredients(response, matcher, nlp)
            #Check if the detected ingredient is valid
            check_valid_ingredients(ingredient, self.ingredients_list)
    
        #Move on to the choosing recipe stage after finishing adding ingredients
        self.ingredients_list = [", ".join(self.ingredients_list)]
        self.current_stage = "choose_recipe_stage"
        print(self.ingredients_list)
        
    ###Step 2.) Choosing Recipe Stage
    def choose_recipe_stage(self):
        #Process the ingredients list and the dataset - tfidf
        df_tfidf, df_vector = perform_tfidf(self.df_food["ingredients"])
        #Apply Cosine similarity to get top 5 matching recipes
        self.top5_recipe = perform_cosine_similarity(df_tfidf, df_vector, self.ingredients_list)
        self.top5_recipe = self.df_food["name"][self.top5_recipe]
        for i,recipe in enumerate(self.top5_recipe):
            print(f"{i + 1}.) {recipe}")
        
        #Ask user to pick their preferred recipe from list
        status = "ongoing"
        while status != "done":
            #Get user response
            response = input("Which recipe would you like to see?:")
            #Clean the response
            response_cleaned = text_preprocessing(response)
            response_cleaned = word_tokenize(response_cleaned)
            response_index = check_recipe_response(response_cleaned, self)
            #Check if the response is valid
            if response_index is not None:
                status = "done"
                self.current_stage = "show_recipe_stage"
                print("Chosen Recipe is: ", self.df_food["name"].iloc[response_index])
            else:
                print("I have not detected a matching recipe. Please try again")

In [428]:
####main
dialogue = DialoguePolicy()
while dialogue.current_stage != "quit":
    if dialogue.current_stage == "ingredients_collection":
        dialogue.choose_ingredient_stage()
    elif dialogue.current_stage == "choose_recipe_stage":
        dialogue.choose_recipe_stage()

Choose your ingredients: mushrooms


Current Ingredients List: ['mushroom']


Choose your ingredients: gravy


Current Ingredients List: ['mushroom', 'gravy']


Choose your ingredients: asparagus


...

Choose your ingredients: done


['mushroom, gravy, asparagus, steak']
1.) Super Steak and Gravy
2.) Steak &amp; Shiitake Fried Rice
3.) Quick French Onion Meatballs
4.) Sylvita's Charro Beans
5.) Beef Bundle Kabobs


Which recipe would you like to see?: I want the second recipe


Chosen Recipe is:  Steak &amp; Shiitake Fried Rice


KeyboardInterrupt: 